In [ ]:
import sys

sys.path.append("../src")

In [ ]:
import pandas as pd

from smogcast.ingest.gios import get_all_stations

In [ ]:
selected_stations_df = pd.read_parquet(
    "../data/raw/gios_capital_stations_poland.parquet"
)

In [ ]:
selected_stations_df.head()

In [ ]:
stations = get_all_stations()

In [ ]:
coordinates_df = pd.DataFrame(
    [
        {
            "station_id": station["Identyfikator stacji"],
            "latitude": float(station["WGS84 φ N"]),
            "longitude": float(station["WGS84 λ E"]),
        }
        for station in stations
    ]
)

In [ ]:
weather_stations_df = selected_stations_df.merge(
    coordinates_df,
    on="station_id",
    how="left",
)

In [ ]:
weather_stations_df[
    [
        "station_id",
        "station_name",
        "city",
        "latitude",
        "longitude",
    ]
]

In [ ]:
weather_stations_df[
    ["latitude", "longitude"]
].isna().sum()

In [ ]:
import pandas as pd

from smogcast.ingest.weather import get_weather_for_location

In [ ]:
station = weather_stations_df.iloc[0]

In [ ]:
station[
    [
        "station_id",
        "station_name",
        "city",
        "latitude",
        "longitude",
    ]
]

In [ ]:
weather_test = get_weather_for_location(
    station["station_id"],
    station["station_name"],
    station["city"],
    station["voivodeship"],
    station["latitude"],
    station["longitude"],
    "2026-08-01",
    "2026-08-03",
)

In [ ]:
type(weather_test)

In [ ]:
len(weather_test)

In [ ]:
weather_test_df = pd.DataFrame(weather_test)

In [ ]:
print(weather_test_df["timestamp_utc"].min())
print(weather_test_df["timestamp_utc"].max())

In [ ]:
from smogcast.ingest.weather import get_weather_for_stations

In [ ]:
station_records = weather_stations_df.to_dict("records")

In [ ]:
len(station_records)

In [ ]:
start_date = "2025-08-20"
end_date = "2026-08-20"

In [ ]:
all_weather, failed_stations = get_weather_for_stations(
    station_records,
    start_date,
    end_date,
)

In [ ]:
len(all_weather)

In [ ]:
len(failed_stations)

In [ ]:
weather_df = pd.DataFrame(all_weather)

In [ ]:
weather_df.head()

In [ ]:
weather_df["timestamp_utc"] = pd.to_datetime(
    weather_df["timestamp_utc"],
    utc=True,
)

In [ ]:
print("Liczba pomiarów:", len(weather_df))
print("Liczba stacji:", weather_df["station_id"].nunique())
print("Najstarszy pomiar:", weather_df["timestamp_utc"].min())
print("Najnowszy pomiar:", weather_df["timestamp_utc"].max())
print("Nieudane stacje:", len(failed_stations))

In [ ]:
weather_df[
    [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]
].isna().sum()

In [ ]:
weather_df.duplicated(
    subset=["station_id", "timestamp_utc"]
).sum()

In [ ]:
weather_df.to_parquet(
    "../data/raw/open_meteo_weather_1y.parquet",
    index=False,
)

In [ ]:
failed_stations_df = pd.DataFrame(
    failed_stations,
    columns=["station_id"],
)

In [ ]:
failed_stations_df.to_csv(
    "../data/raw/open_meteo_failed_stations_1y.csv",
    index=False,
)

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))

from smogcast.ingest.weather import get_weather_for_stations

In [ ]:
CURRENT_WEATHER_PATH = "../data/raw/open_meteo_weather_1y.parquet"
OLDER_WEATHER_PATH = "../data/raw/open_meteo_weather_older.parquet"
FULL_WEATHER_PATH = "../data/raw/open_meteo_weather_3y.parquet"

START_DATE = "2023-08-20"
END_DATE = "2025-08-19"

In [ ]:
current_weather = pd.read_parquet(CURRENT_WEATHER_PATH)

current_weather.head()

In [ ]:
station_records_df = (
    current_weather[
        [
            "station_id",
            "station_name",
            "city",
            "voivodeship",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

station_records = station_records_df.to_dict(
    orient="records"
)

print("Liczba stacji:", len(station_records))

In [ ]:
older_weather_records, failed_stations = get_weather_for_stations(
    station_records=station_records,
    start_date=START_DATE,
    end_date=END_DATE,
)

In [ ]:
older_weather = pd.DataFrame(older_weather_records)

older_weather.shape

In [ ]:
older_weather["timestamp_utc"] = pd.to_datetime(
    older_weather["timestamp_utc"],
    utc=True,
)

print(
    older_weather["timestamp_utc"].min(),
    "→",
    older_weather["timestamp_utc"].max(),
)

In [ ]:
print("Nieudane stacje:", len(failed_stations))
print(failed_stations)

In [ ]:
older_weather.to_parquet(
    "../data/raw/open_meteo_weather_older.parquet",
    index=False,
)

In [ ]:
current_weather["timestamp_utc"] = pd.to_datetime(
    current_weather["timestamp_utc"],
    utc=True,
)

In [ ]:
weather_3y = pd.concat(
    [
        older_weather,
        current_weather,
    ],
    ignore_index=True,
)

In [ ]:
before = len(weather_3y)

weather_3y = (
    weather_3y
    .drop_duplicates(
        subset=[
            "station_id",
            "timestamp_utc",
        ]
    )
    .reset_index(drop=True)
)

after = len(weather_3y)

print("Przed:", before)
print("Po:", after)
print("Duplikaty usunięte:", before - after)

In [ ]:
print(
    "Zakres:",
    weather_3y["timestamp_utc"].min(),
    "→",
    weather_3y["timestamp_utc"].max(),
)

print("Rekordów:", len(weather_3y))
print("Stacji:", weather_3y["station_id"].nunique())

In [ ]:
weather_3y.to_parquet(
    "../data/raw/open_meteo_weather_3y.parquet",
    index=False,
)